# 02 — Exploratory Data Analysis (EDA)

**Objective:** Perform a thorough exploratory analysis of the clinical trials dataset to understand variable distributions, relationships with the target (`abandoned`), class imbalance, and identify signal features for modeling.

**Sections:**
1. Setup & Data Loading
2. Univariate Analysis
3. Bivariate Analysis
4. Class Imbalance Analysis
5. Synthesis

---
## 0. Setup & Data Loading

We import libraries, configure plotting defaults, and load the dataset.  
We also define column groups (numerical, categorical, binary) up front so every subsequent section can loop over them consistently.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# ── Plot style ───────────────────────────────────────────────────────────────
# We use a clean seaborn style with a muted palette suitable for publication.
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 100,
    'figure.figsize': (12, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# ── Color palette for the two target classes ─────────────────────────────────
# Consistent colors across the entire notebook:
#   class 0 (completed) → teal,  class 1 (abandoned) → coral
TARGET_COLORS = {0: '#2a9d8f', 1: '#e76f51', '0': '#2a9d8f', '1': '#e76f51', '0.0': '#2a9d8f', '1.0': '#e76f51'}
TARGET_PALETTE = ['#2a9d8f', '#e76f51']

print('Libraries loaded ✓')

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────────
df = pd.read_csv('../data/dataset.csv')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Target distribution:\n{df["abandoned"].value_counts()}')
df.head()

In [ ]:
# ── Column groups ────────────────────────────────────────────────────────────
# Defining groups explicitly avoids accidental inclusion/exclusion of columns
# and makes it easy to iterate in each analysis section.

NUMERICAL_COLS = [
    'enrollment_count', 'n_arms', 'n_primary_outcomes',
    'n_secondary_outcomes', 'outcomes_ratio', 'n_locations', 'n_collaborators'
]

CATEGORICAL_COLS = [
    'phase', 'sponsor_type', 'intervention_type',
    'allocation', 'masking', 'primary_purpose'
]

BINARY_COLS = ['has_dmc', 'is_multicenter', 'has_us_site']

TARGET = 'abandoned'

print(f'Numerical  ({len(NUMERICAL_COLS)}): {NUMERICAL_COLS}')
print(f'Categorical ({len(CATEGORICAL_COLS)}): {CATEGORICAL_COLS}')
print(f'Binary     ({len(BINARY_COLS)}): {BINARY_COLS}')
print(f'Target     : {TARGET}')

In [ ]:
# ── Quick overview: dtypes and missing values ────────────────────────────────
# This gives us a first sense of data quality before diving into each section.

info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.count(),
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2),
    'n_unique': df.nunique()
})
info_df

---
## 1. Univariate Analysis

We examine each variable in isolation to understand its distribution, central tendency, spread, and potential outliers.  
This section is split into three subsections matching the variable types:

- **1.1 Numerical variables** — histograms, boxplots, descriptive stats
- **1.2 Categorical variables** — frequency bar charts, rare-modality flagging
- **1.3 Binary variables** — frequency counts and bar charts

### 1.1 Numerical Variables

For each numerical feature we produce:
- A **histogram** (with KDE overlay) to see the shape of the distribution.
- A **boxplot** to quickly spot outliers and the interquartile range.
- A **descriptive statistics table** (mean, median, std, Q1, Q3, min, max).

Many clinical-trial variables are right-skewed (e.g. `enrollment_count`, `n_locations`) because a few large multi-centre trials pull the tail.

In [ ]:
# ── Histogram + Boxplot for every numerical variable ─────────────────────────
# We pair a histogram and a boxplot side by side so we can see both the
# distribution shape and outlier structure at a glance.

for col in NUMERICAL_COLS:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Histogram with KDE
    sns.histplot(df[col].dropna(), bins=50, kde=True, ax=axes[0],
                 color='#264653', edgecolor='white', linewidth=0.5)
    axes[0].set_title(f'Distribution of {col}')
    axes[0].set_xlabel(col)
    axes[0].set_ylabel('Count')

    # Boxplot
    sns.boxplot(x=df[col].dropna(), ax=axes[1], color='#2a9d8f',
                flierprops=dict(marker='o', markersize=3, alpha=0.4))
    axes[1].set_title(f'Boxplot of {col}')
    axes[1].set_xlabel(col)

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Descriptive statistics table ─────────────────────────────────────────────
# We compute the standard summary statistics asked for: mean, median, std,
# Q1, Q3, min, max.  Rounding to 2 decimals keeps the table readable.

desc_stats = df[NUMERICAL_COLS].describe().T[['mean', '50%', 'std', '25%', '75%', 'min', 'max']]
desc_stats.columns = ['Mean', 'Median', 'Std', 'Q1', 'Q3', 'Min', 'Max']
desc_stats = desc_stats.round(2)

# Also add missing count for context
desc_stats['Missing'] = df[NUMERICAL_COLS].isnull().sum().values
desc_stats['Missing %'] = (df[NUMERICAL_COLS].isnull().sum().values / len(df) * 100).round(2)

desc_stats.style.background_gradient(cmap='YlOrRd', subset=['Std'], axis=0)

### 1.2 Categorical Variables

For each categorical feature:
- A **bar chart** of modality frequencies (sorted descending).
- **Flagging** of rare modalities that appear in < 1 % of rows — these may need grouping or special handling during modeling.

In [ ]:
# ── Bar chart of frequencies + rare modality flagging ────────────────────────
# We flag modalities below 1 % frequency in red on the bar chart so they
# stand out visually.  We also print a summary table below.
# NOTE: We replace NaN with the string 'Missing' and cast to string so matplotlib 
# can use it as a categorical bar label without TypeError.

RARE_THRESHOLD = 0.01  # 1 %

rare_modalities_summary = {}  # collect for the synthesis

for col in CATEGORICAL_COLS:
    # fillna ensures NaN becomes a proper string label for plotting
    counts = df[col].fillna('Missing').astype(str).value_counts()
    pcts = counts / len(df)
    rare_mask = pcts < RARE_THRESHOLD

    # Build color list: rare = coral, normal = teal
    colors = ['#e76f51' if rare_mask[cat] else '#2a9d8f' for cat in counts.index]

    fig, ax = plt.subplots(figsize=(12, max(4, len(counts) * 0.45)))
    bars = ax.barh(counts.index.astype(str), counts.values, color=colors,
                   edgecolor='white', linewidth=0.5)

    # Annotate counts on bars
    for bar, cnt, pct in zip(bars, counts.values, pcts.values):
        ax.text(bar.get_width() + max(counts) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{cnt}  ({pct:.1%})', va='center', fontsize=9)

    ax.set_title(f'Frequency of {col}  (red = < 1 %)', fontsize=13)
    ax.set_xlabel('Count')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    # Collect rare modalities
    rare_cats = list(pcts[rare_mask].index)
    if rare_cats:
        rare_modalities_summary[col] = rare_cats
        print(f'  ⚠  Rare modalities (< 1 %): {rare_cats}')
    print()

In [ ]:
# ── Summary of all rare modalities across categorical variables ──────────────
print('Rare modalities (< 1 %) summary:')
print('=' * 60)
for col, cats in rare_modalities_summary.items():
    print(f'  {col}: {cats}')

### 1.3 Binary Variables

Binary variables have only two values (0/1).  
We show frequency counts and simple bar charts to assess the balance of each binary feature.

In [ ]:
# ── Binary variable frequency counts and bar charts ──────────────────────────
# For binary features we display both the raw count and percentage.
# Note: has_dmc may have missing values (NaN for trials where DMC info is absent).

fig, axes = plt.subplots(1, len(BINARY_COLS), figsize=(5 * len(BINARY_COLS), 4))

for i, col in enumerate(BINARY_COLS):
    ax = axes[i]
    # fillna to handle any NaN in binary columns for plotting, and astype(str) to avoid float/str sort issues
    counts = df[col].fillna('Missing').astype(str).value_counts().sort_index()
    labels = [str(k) for k in counts.index]
    colors = ['#2a9d8f' if str(k) in ('0', '0.0') else '#e76f51' if str(k) in ('1', '1.0') else '#adb5bd'
              for k in counts.index]

    bars = ax.bar(labels, counts.values, color=colors, edgecolor='white', linewidth=0.8)

    # Annotate
    for bar, cnt in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + len(df) * 0.005,
                f'{cnt}\n({cnt/len(df):.1%})', ha='center', va='bottom', fontsize=10)

    ax.set_title(col, fontsize=13)
    ax.set_ylabel('Count')

plt.suptitle('Binary Variables — Frequency Counts', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Print frequency table
for col in BINARY_COLS:
    print(f'\n{col}:')
    vc = df[col].fillna('Missing').astype(str).value_counts()
    for val, cnt in vc.items():
        print(f'  {val}: {cnt} ({cnt/len(df):.2%})')

---
## 2. Bivariate Analysis

We now study how each feature relates to the target variable `abandoned`, and how numerical features relate to each other.

- **2.1 Numerical × Target** — side-by-side boxplots for each class
- **2.2 Categorical × Target** — crosstab + abandonment-rate bar charts
- **2.3 Numerical × Numerical** — correlation matrix heatmap

### 2.1 Numerical × Target

Side-by-side boxplots let us visually compare the distribution of each numerical variable between completed (0) and abandoned (1) trials.  
Differences in median or spread can indicate a feature's predictive potential.

In [ ]:
# ── Side-by-side boxplots: each numerical variable split by target ────────────
# We use the consistent teal/coral palette for class 0/1.

for col in NUMERICAL_COLS:
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.boxplot(
        data=df, x=TARGET, y=col, ax=ax,
        palette=TARGET_COLORS,
        flierprops=dict(marker='o', markersize=3, alpha=0.3),
        width=0.5
    )
    ax.set_title(f'{col}  by  {TARGET}', fontsize=13)
    ax.set_xticklabels(['Completed (0)', 'Abandoned (1)'])
    plt.tight_layout()
    plt.show()

### 2.2 Categorical × Target

For every categorical feature we build:
- A **crosstab** (count of each modality × target class).
- A **bar chart of the abandonment rate** per modality, with a red dashed line showing the overall abandonment rate for comparison.

Modalities with abandonment rates well above or below the global average are potential signal features.

In [ ]:
# ── Crosstab + abandonment-rate bar chart per categorical variable ────────────
# We use fillna('Missing') to include NaN as a visible modality in crosstabs.

global_abandon_rate = df[TARGET].mean()
print(f'Global abandonment rate: {global_abandon_rate:.2%}\n')

for col in CATEGORICAL_COLS:
    col_filled = df[col].fillna('Missing').astype(str)

    # Crosstab
    ct = pd.crosstab(col_filled, df[TARGET], margins=True)
    ct.columns = ['Completed (0)', 'Abandoned (1)', 'Total']
    ct['Abandon Rate'] = (ct['Abandoned (1)'] / ct['Total']).map('{:.2%}'.format)
    display(ct)

    # Abandonment rate per modality
    abandon_rate = pd.DataFrame({'target': df[TARGET].values, 'col': col_filled.values})
    abandon_rate = abandon_rate.groupby('col')['target'].mean().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(12, max(4, len(abandon_rate) * 0.5)))
    bars = ax.barh(abandon_rate.index.astype(str), abandon_rate.values,
                   color='#264653', edgecolor='white', linewidth=0.5)

    # Global average line
    ax.axvline(global_abandon_rate, color='#e76f51', linestyle='--', linewidth=2,
               label=f'Global rate ({global_abandon_rate:.1%})')

    # Annotate bars
    for bar, rate in zip(bars, abandon_rate.values):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{rate:.1%}', va='center', fontsize=9)

    ax.set_title(f'Abandonment rate by {col}', fontsize=13)
    ax.set_xlabel('Abandonment rate')
    ax.set_xlim(0, min(1.0, abandon_rate.max() + 0.1))
    ax.legend(loc='lower right')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    print()

### 2.3 Numerical × Numerical — Correlation Matrix

A heatmap of Pearson correlations between numerical variables.  
We flag pairs with |corr| > 0.7 — these may cause multicollinearity problems in certain models and might need to be addressed (e.g. dropping one, or using PCA).

In [ ]:
# ── Correlation heatmap ──────────────────────────────────────────────────────

corr = df[NUMERICAL_COLS].corr()

# Create a mask for the upper triangle (avoid duplication)
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, linewidths=0.5,
    square=True, ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Pearson Correlation Matrix — Numerical Variables', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Flag highly correlated pairs (|corr| > 0.7) ──────────────────────────────
# We extract the upper triangle and filter for strong correlations.

HIGH_CORR_THRESHOLD = 0.7

upper = corr.where(np.triu(np.ones_like(corr, dtype=bool), k=1))
high_corr_pairs = [
    (col1, col2, upper.loc[col1, col2])
    for col1 in upper.index
    for col2 in upper.columns
    if abs(upper.loc[col1, col2]) > HIGH_CORR_THRESHOLD
]

if high_corr_pairs:
    print(f'⚠  Pairs with |corr| > {HIGH_CORR_THRESHOLD}:')
    for c1, c2, r in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True):
        print(f'   {c1}  ↔  {c2}  :  r = {r:.3f}')
else:
    print(f'✓  No pairs with |corr| > {HIGH_CORR_THRESHOLD}')

---
## 3. Class Imbalance Analysis

The target variable `abandoned` is imbalanced (~86 % completed vs ~14 % abandoned).  
This section quantifies the imbalance and compares feature distributions across the two classes to identify the strongest "signal features".

Subsections:
- **3.1** Exact class ratio + pie chart + bar chart
- **3.2** Descriptive stats split by class
- **3.3** Signal features — variables where class distributions differ most

### 3.1 Class Distribution

In [ ]:
# ── Exact class ratio ────────────────────────────────────────────────────────

class_counts = df[TARGET].value_counts().sort_index()
class_pcts = (class_counts / len(df) * 100).round(2)

print('Class distribution:')
print(f'  Class 0 (Completed) : {class_counts[0]:,}  ({class_pcts[0]:.2f} %)')
print(f'  Class 1 (Abandoned) : {class_counts[1]:,}  ({class_pcts[1]:.2f} %)')
print(f'  Imbalance ratio     : {class_counts[0] / class_counts[1]:.1f} : 1')

In [ ]:
# ── Pie chart + bar chart of class distribution ──────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(
    class_counts.values,
    labels=['Completed (0)', 'Abandoned (1)'],
    colors=TARGET_PALETTE,
    autopct='%1.1f%%',
    startangle=90,
    explode=(0, 0.06),
    textprops={'fontsize': 12},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Class Distribution — Pie Chart', fontsize=14)

# Bar chart
bars = axes[1].bar(
    ['Completed (0)', 'Abandoned (1)'],
    class_counts.values,
    color=TARGET_PALETTE,
    edgecolor='white', linewidth=1.5, width=0.5
)
for bar, cnt, pct in zip(bars, class_counts.values, class_pcts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                 f'{cnt:,}\n({pct:.1f} %)', ha='center', va='bottom', fontsize=12)
axes[1].set_title('Class Distribution — Bar Chart', fontsize=14)
axes[1].set_ylabel('Count')

plt.suptitle('Target Class Imbalance', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### 3.2 Descriptive Statistics by Class

We compute separate descriptive statistics tables for class 0 (completed) and class 1 (abandoned).  
Comparing medians and means across the two classes helps spot features whose distributions shift between outcomes.

In [ ]:
# ── Descriptive stats for Class 0 (Completed) and Class 1 (Abandoned) ───────

df_class0 = df[df[TARGET] == 0]
df_class1 = df[df[TARGET] == 1]

def build_desc_table(subset, label):
    """Build a clean descriptive stats table for a subset."""
    desc = subset[NUMERICAL_COLS].describe().T[['mean', '50%', 'std', '25%', '75%', 'min', 'max']]
    desc.columns = ['Mean', 'Median', 'Std', 'Q1', 'Q3', 'Min', 'Max']
    desc = desc.round(2)
    desc.index.name = f'{label} (n={len(subset):,})'
    return desc

desc_class0 = build_desc_table(df_class0, 'Class 0 — Completed')
desc_class1 = build_desc_table(df_class1, 'Class 1 — Abandoned')

print('▸ Class 0 — Completed trials')
display(desc_class0.style.background_gradient(cmap='Blues', axis=0))

print('\n▸ Class 1 — Abandoned trials')
display(desc_class1.style.background_gradient(cmap='Oranges', axis=0))

In [ ]:
# ── Side-by-side comparison: median shift between classes ─────────────────────
# A quick comparison of medians helps surface the largest distributional shifts.

comparison = pd.DataFrame({
    'Median (class 0)': df_class0[NUMERICAL_COLS].median(),
    'Median (class 1)': df_class1[NUMERICAL_COLS].median(),
    'Mean (class 0)': df_class0[NUMERICAL_COLS].mean().round(2),
    'Mean (class 1)': df_class1[NUMERICAL_COLS].mean().round(2),
})
comparison['Median Δ'] = (comparison['Median (class 1)'] - comparison['Median (class 0)']).round(2)
comparison['Median Δ %'] = (
    (comparison['Median Δ'] / comparison['Median (class 0)'].replace(0, np.nan)) * 100
).round(1)

comparison.style.background_gradient(cmap='RdYlGn_r', subset=['Median Δ %'], axis=0)

### 3.3 Signal Features

We identify features whose distributions differ most between the two classes.  
- For **numerical** features: Mann-Whitney U test (non-parametric, robust to skew).  
- For **categorical** features: chi-squared test on the crosstab.  
- For **binary** features: chi-squared test.

We rank by effect size / test statistic and flag the strongest signals.

In [ ]:
# ── Signal detection: statistical tests ──────────────────────────────────────
# We use non-parametric tests because most numerical variables are skewed.

signal_results = []

# --- Numerical features: Mann-Whitney U ---
for col in NUMERICAL_COLS:
    g0 = df_class0[col].dropna()
    g1 = df_class1[col].dropna()
    if len(g0) > 0 and len(g1) > 0:
        stat, pval = stats.mannwhitneyu(g0, g1, alternative='two-sided')
        # Rank-biserial correlation as effect size
        r_effect = 1 - (2 * stat) / (len(g0) * len(g1))
        signal_results.append({
            'Feature': col,
            'Type': 'Numerical',
            'Test': 'Mann-Whitney U',
            'Statistic': round(stat, 1),
            'p-value': pval,
            'Effect Size': round(abs(r_effect), 4),
            'Significant (p<0.05)': pval < 0.05
        })

# --- Categorical & Binary features: Chi-squared ---
# We use fillna('Missing') to include NaN as a category in the crosstab,
# ensuring NaN rows are not silently dropped from the test.
for col in CATEGORICAL_COLS + BINARY_COLS:
    ct = pd.crosstab(df[col].fillna('Missing').astype(str), df[TARGET])
    if ct.shape[0] >= 2 and ct.shape[1] >= 2:
        chi2, pval, dof, expected = stats.chi2_contingency(ct)
        # Cramér's V as effect size
        n = ct.sum().sum()
        k = min(ct.shape) - 1
        cramers_v = np.sqrt(chi2 / (n * k)) if k > 0 else 0
        signal_results.append({
            'Feature': col,
            'Type': 'Categorical' if col in CATEGORICAL_COLS else 'Binary',
            'Test': 'Chi-squared',
            'Statistic': round(chi2, 1),
            'p-value': pval,
            'Effect Size': round(cramers_v, 4),
            'Significant (p<0.05)': pval < 0.05
        })

signal_df = pd.DataFrame(signal_results).sort_values('Effect Size', ascending=False)
signal_df.index = range(1, len(signal_df) + 1)
signal_df.index.name = 'Rank'

# Pandas >= 2.1.0 uses .map instead of .applymap
style_func = getattr(signal_df.style, 'map', getattr(signal_df.style, 'applymap', None))
if style_func:
    style_func(
        lambda v: 'background-color: #d4edda' if v is True else
                  ('background-color: #f8d7da' if v is False else ''),
        subset=['Significant (p<0.05)']
    )
else:
    pass # fallback if neither exist

In [ ]:
# ── Visual ranking of signal features by effect size ─────────────────────────

fig, ax = plt.subplots(figsize=(12, max(5, len(signal_df) * 0.45)))

colors_signal = [
    '#e76f51' if sig else '#adb5bd'
    for sig in signal_df['Significant (p<0.05)'].values
]

ax.barh(signal_df['Feature'], signal_df['Effect Size'], color=colors_signal,
        edgecolor='white', linewidth=0.5)

for i, (feat, es) in enumerate(zip(signal_df['Feature'], signal_df['Effect Size'])):
    ax.text(es + 0.005, i, f'{es:.4f}', va='center', fontsize=9)

ax.set_xlabel('Effect Size', fontsize=12)
ax.set_title('Signal Features — Ranked by Effect Size\n(coral = statistically significant at p < 0.05)',
             fontsize=13)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# List top signals
top_signals = signal_df[signal_df['Significant (p<0.05)'] == True]['Feature'].tolist()
print(f'\n✓ Statistically significant signal features: {top_signals}')

---
## 4. Synthesis

### Key Findings

This section consolidates the most important observations from the EDA.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SYNTHESIS — Programmatic summary of key findings
# ══════════════════════════════════════════════════════════════════════════════

print('=' * 70)
print('                      EDA SYNTHESIS')
print('=' * 70)

# ── 1. Missing values ────────────────────────────────────────────────────────
print('\n📌  MISSING VALUES')
print('-' * 40)
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    for col_name, cnt in missing.items():
        print(f'   {col_name}: {cnt} missing ({cnt/len(df):.1%})')
else:
    print('   No missing values detected.')

# ── 2. Class imbalance ───────────────────────────────────────────────────────
print(f'\n📌  CLASS IMBALANCE')
print('-' * 40)
print(f'   Completed (0): {class_counts[0]:,} ({class_pcts[0]:.2f} %)')
print(f'   Abandoned (1): {class_counts[1]:,} ({class_pcts[1]:.2f} %)')
print(f'   Ratio: {class_counts[0]/class_counts[1]:.1f} : 1')
print(f'   → Consider: SMOTE, class weights, stratified splits')

# ── 3. Signal features ───────────────────────────────────────────────────────
print(f'\n📌  SIGNAL FEATURES (statistically significant, ranked by effect size)')
print('-' * 40)
for _, row in signal_df[signal_df['Significant (p<0.05)']].iterrows():
    print(f'   {row["Feature"]:25s}  effect = {row["Effect Size"]:.4f}  '
          f'(p = {row["p-value"]:.2e})  [{row["Test"]}]')

# ── 4. Notable outliers ──────────────────────────────────────────────────────
print(f'\n📌  NOTABLE OUTLIERS')
print('-' * 40)
for col in NUMERICAL_COLS:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper_bound)).sum()
    if n_outliers > 0:
        pct_outliers = n_outliers / df[col].notna().sum() * 100
        print(f'   {col}: {n_outliers} outliers ({pct_outliers:.1f} %)  '
              f'[range: {df[col].min():.0f} – {df[col].max():.0f}]')

# ── 5. Correlations to watch ─────────────────────────────────────────────────
print(f'\n📌  CORRELATIONS TO WATCH (|r| > {HIGH_CORR_THRESHOLD})')
print('-' * 40)
if high_corr_pairs:
    for c1, c2, r in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True):
        print(f'   {c1}  ↔  {c2}  :  r = {r:.3f}')
    print('   → Consider dropping one feature or using PCA.')
else:
    print(f'   No pairs exceed |r| > {HIGH_CORR_THRESHOLD}.')

# ── 6. Rare modalities ───────────────────────────────────────────────────────
print(f'\n📌  RARE MODALITIES (< 1 % frequency)')
print('-' * 40)
if rare_modalities_summary:
    for col_name, cats in rare_modalities_summary.items():
        print(f'   {col_name}: {cats}')
    print('   → Consider grouping into an "Other" category for modeling.')
else:
    print('   None detected.')

print('\n' + '=' * 70)
print('End of EDA.  Proceed to feature engineering & modeling.')
print('=' * 70)